# **Using qc-bench**

In [1]:
from qc_bench import Judge, annotate_csv

import logging

logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,
    handlers=[
        logging.FileHandler("log.txt", mode="a", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)

## **Example Data**

Consider the following example data:
- `src` is our source text,
- `mt` is our machine translation that we want to rate in terms of quality,
- `src_lang` is the language of the source text,
- and `mt_lang` is the target language of the machine translation.

In [2]:
src = (
    "The lights are dimmable, but I use the strongest setting only. I have not made use of the timer, preferring to turn them on and off myself. "
    "I can see this feature as useful in an office setting with houseplants or if on vacation."
)
mt = (
    "Die Lichter sind dimmbar, aber ich benutze nur die stärkste Einstellung. Ich benutze den Timer nicht, sondern schalte ihn lieber selbst ein und aus. "
    "Ich sehe diese Funktion als nützlich im Büro mit Zimmerpflanzen oder im Urlaub."
)
src_lang = "English"
mt_lang = "German"

## **Setting up OpenAI, Anthropic, Google and Ollama Models**

The `Judge` constructor accepts four possible arguments:
- ```python
  openai: str | bool | None = None
  ```
- ```python
  anthropic: str | bool | None = None
  ```
- ```python
  google: str | bool | None = None
  ```
- ```python
  ollama: str | bool | None = None
  ```

All of these arguments can either be `str`, `bool`, or `None` (default):
- If `None` or `True` the constructor will try to read the corresponding API keys from your environment and throw an error if it can't find it in the environment.
- If `str` the constructor expects a valid API key in string format (with the exception of `ollama` which expects a valid model identifier).
- If `False` the specific model will be disabled in the judge.

### **Example: Reading the API key for OpenAI model access from the environment**

In [3]:
judge_no_env = Judge(openai=True, anthropic=False, google=False, ollama=False)

ERROR:qc_bench._judge:Could not get an API key for OpenAI! Searched for 'OPENAI_API_KEY'.


RuntimeError: Could not get an API key for OpenAI! Searched for 'OPENAI_API_KEY'.

An error is thrown because the environment does not contain an API key for OpenAI models.

**Please never commit your environments or API keys!**

### **Example: Manually specifying the API key for OpenAI model access**

In [4]:
judge_invalid_api_key = Judge(
    openai="YOUR_API_KEY", anthropic=False, google=False, ollama=False
)

Providing an invalid API key (as here `"YOUR_API_KEY"`) will not throw an error upon creating a judge, but will obviously lead to failing requests.

**Please never commit your environments or API keys!**

### **Example: Setting up Ollama model access**

In [5]:
judge_gemma4 = Judge(openai=False, anthropic=False, google=False, ollama=True)

You can setup the judge to use Ollama by providing `ollama=True` which will use the default model (currently `gemma4:e4b`) for rating translations.

In [6]:
judge_gemma4.ollama_model

'gemma4:e4b'

### **Example: Setting up a specific Ollama model**

In [7]:
judge_qwen = Judge(openai=False, anthropic=False, google=False, ollama="qwen3.8:27b")

You can request a specific Ollama model by providing the model identifier as the `ollama` argument.

## **Scoring a Translation**

We can score a specific translation by using the `Judge.score()` function. We use the example data from above here.

In [8]:
jr = judge_qwen.score(src=src, mt=mt, src_lang=src_lang, mt_lang=mt_lang)

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:qc_bench._judge:Successfully got a valid response after retry 0 for one query.


In [9]:
type(jr)

qc_bench._judge.JudgeResult

The `Judge.score()` function will return a `JudgeResult` which has four attributes (see also the docs on `JudgeResult` for details):
- `openai`
- `anthropic`
- `google`
- `ollama`

In [10]:
jr.openai is None

True

The `openai`, `anthropic` and `google` attributes will all be `None` because we did not setup the judge to use those models!

In [11]:
jr.ollama is None

False

The `ollama` attribute will not be `None` because we setup Ollama.

In [12]:
type(jr.ollama)

qc_bench._judge.JudgeModelResult

Any attribute of a `JudgeResult` will be a `JudgeModelResult` (if not `None`) - please also refer to the docs on `JudgeModelResult` for more details!

In [13]:
jr.ollama.model  # ty: ignore[unresolved-attribute]

'qwen3.8:27b'

Each `JudgeModelResult` contains the model.

In [14]:
jr.ollama.prompt  # ty: ignore[unresolved-attribute]

'You are an annotator for the quality of machine translation. Your task is to\n            identify errors and assess the quality of the translation.\n            Based on the source segment, human-generated reference translation, and machine\n            translation surrounded with triple backticks, identify error types in the\n            translation and classify them. The categories of errors are: accuracy\n            (addition, mistranslation, omission, untranslated text), fluency (character\n            encoding, grammar, inconsistency, punctuation, register, spelling), style\n            (awkward), terminology (inappropriate for context, inconsistent use),\n            non-translation, other, or no-error.\n            Each error is classified as one of three severities: critical, major, and minor.\n            Critical errors inhibit comprehension of the text. Major errors disrupt the\n            flow, but what the text is trying to say is still understandable. Minor errors\n  

Each `JudgeModelResult` contains the prompt that was used.

In [15]:
jr.ollama.status  # ty: ignore[unresolved-attribute]

'ok'

Each `JudgeModelResult` contains the status of the response which can be `"ok"` or `"error"`, please refer to the log to debug errors.

In [16]:
jr.ollama.parameters  # ty: ignore[unresolved-attribute]

{'num_predict': '2048', 'seed': '1337'}

Each `JudgeModelResult` contains the additional parameters that were used with the model.

In [17]:
jr.ollama.response  # ty: ignore[unresolved-attribute]

'model=\'qwen3.8:27b\' created_at=\'2026-09-14T21:42:37.8608256Z\' done=True done_reason=\'stop\' total_duration=132000915700 load_duration=6874396100 prompt_eval_count=394 prompt_eval_duration=1194658000 eval_count=1608 eval_duration=123276474000 message=Message(role=\'assistant\', content=\'{\\n  "source": "The lights are dimmable, but I use the strongest setting only. I have not made use of the timer, preferring to turn them on and off myself. I can see this feature as useful in an office setting with houseplants or if on vacation.",\\n  "machine_translation": "Die Lichter sind dimmbar, aber ich benutze nur die stärkste Einstellung. Ich benutze den Timer nicht, sondern schalte ihn lieber selbst ein und aus. Ich sehe diese Funktion als nützlich im Büro mit Zimmerpflanzen oder im Urlaub.",\\n  "quality_estimation_value": 0.72,\\n  "errors": [\\n    {\\n      "error_type": "accuracy",\\n      "error_subtype": "mistranslation",\\n      "error_severity": "major",\\n      "description": "

Each `JudgeModelResult` with status `"ok"` contains the full model response.

In [18]:
type(jr.ollama.quality_estimation)  # ty: ignore[unresolved-attribute]

qc_bench._translation.QualityEstimation

Each `JudgeModelResult` with status `"ok"` contains the parsed `QualityEstimation`.

In [19]:
jr.ollama.quality_estimation_dict  # ty: ignore[unresolved-attribute]

{'source': 'The lights are dimmable, but I use the strongest setting only. I have not made use of the timer, preferring to turn them on and off myself. I can see this feature as useful in an office setting with houseplants or if on vacation.',
 'machine_translation': 'Die Lichter sind dimmbar, aber ich benutze nur die stärkste Einstellung. Ich benutze den Timer nicht, sondern schalte ihn lieber selbst ein und aus. Ich sehe diese Funktion als nützlich im Büro mit Zimmerpflanzen oder im Urlaub.',
 'quality_estimation_value': 0.72,
 'errors': [{'error_type': 'accuracy',
   'error_subtype': 'mistranslation',
   'error_severity': 'major',
   'description': "The pronoun 'ihn' (masculine singular) refers to 'den Timer', but the source 'them' refers to 'the lights' (die Lichter, feminine plural). The correct translation should be 'sie' (referring to the lights). The machine translation says 'I prefer to turn the timer on and off myself' instead of 'I prefer to turn them (the lights) on and off

Each `JudgeModelResult` with status `"ok"` contains the parsed `QualityEstimation` as python native dictionary.

In [20]:
jr.ollama.score  # ty: ignore[unresolved-attribute]

0.72

Each `JudgeModelResult` with status `"ok"` contains the parsed `QualityEstimation` score.

## **Scoring a Translation File**

We can also streamline the scoring of translations by submitting a `csv` file with the columns `src`, `mt`, `src_lang`, `mt_lang` and a judge to the utility function `annotate_csv()`.

In [21]:
pl_df, list_jr = annotate_csv("../data/test.csv", judge=judge_gemma4)

INFO:qc_bench._util:Reading file ../data/test.csv...
INFO:qc_bench._util:Successfully read file ../data/test.csv!
Annotating ../data/test.csv...:   0%|          | 0/1 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:qc_bench._judge:Successfully got a valid response after retry 0 for one query.
Annotating ../data/test.csv...: 100%|██████████| 1/1 [00:26<00:00, 26.84s/it]
INFO:qc_bench._util:Finished annotation of ../data/test.csv!


In [22]:
type(pl_df)

polars.dataframe.frame.DataFrame

In [23]:
pl_df

src,mt,src_lang,mt_lang,score_openai,score_anthropic,score_google,score_ollama_gemma4:e4b
str,str,str,str,f64,f64,f64,f64
"""The lights are dimmable, but I…","""Die Lichter sind dimmbar, aber…","""English""","""German""",NaN,NaN,NaN,0.97


This returns a polars DataFrame...

In [24]:
type(list_jr)

list

In [25]:
type(list_jr[0])

qc_bench._judge.JudgeResult

...and a list of `JudgeResult` containing the raw results.